# Построение кривых общего поведения точек на диаграммах рассеяния.

Подключение библиотек

In [26]:
%matplotlib agg #отключает автоматический вывод графиков
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import seaborn.objects as so
import numpy as np
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr, spearmanr, kendalltau
from scipy import stats
from scipy.stats import linregress

Чтение таблицы из файла с помощью pandas

In [27]:
file_path = "D:\\AI\\Stroop test analysis\\dataset_mental_report_new.xlsx"
data = pd.read_excel(file_path)

Вывод таблицы

In [28]:
data

,Unnamed: 0,num,marker_data_id,apway_id,context,person,gender,old,old_group,stroop_duration,...,HF_per,FR,SN,Q1,Q2,Q3,Q4,Q_common,adapt_risk,Q_edit
0,0,1,65787,1220_11316,Sofia,201,М,35,22_35,268,...,23.18,2.49,-1.03,100.0,0.0,0.0,0.0,Q1,1,0
1,1,2,65810,1226_11325,Sofia,105,М,56,36_60,836,...,74.89,0.47,1.74,0.0,78.4,21.6,0.0,Q2,6,0
2,2,3,66190,1242_11509,Sofia,106,М,60,36_60,602,...,26.65,-0.03,0.85,2.0,62.7,33.3,2.0,Q3,6,0
3,3,4,66519,1258_11745,Sofia,001test,Ж,23,22_35,136,...,5.84,-0.42,0.65,0.0,50.0,50.0,0.0,Q3,7,0
4,4,5,66555,1256_11772,Sofia,107,Ж,77,60_PLUS,472,...,70.93,0.66,2.08,0.0,100.0,0.0,0.0,Q2,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414,414,417,264116,5645_46265,1,eka4,Ж,60,36_60,253,...,33.03,0.36,1.56,0.0,68.8,31.2,0.0,Q2,6,0
415,415,418,264139,5623_46271,1,eka,Ж,29,22_35,139,...,22.11,-0.55,1.18,0.0,25.0,75.0,0.0,Q3,7,0
416,416,419,264233,5642_46347,1,eka3,Ж,45,36_60,236,...,15.65,1.24,0.88,0.0,100.0,0.0,0.0,Q2,5,0
417,417,420,264269,5623_46352,1,eka,Ж,29,22_35,128,...,31.79,1.02,0.67,0.0,100.0,0.0,0.0,Q2,5,0


Выбор только нужных данных

In [29]:
necessary = data[['gender','old']]
necessary.head()

,gender,old
0,М,35
1,М,56
2,М,60
3,Ж,23
4,Ж,77


Фильтрация данных и разделение по полу.(pandas)

In [30]:
men = necessary[necessary['gender']=='М']['old']
women = necessary[necessary['gender']=='Ж']['old']

In [34]:
mask_covid = data['context'] == 'COVID'  #маски для фильтрации это булев массив из 0 и 1
mask_autism = data['context'] == 'Autism'
mask_rehab = data['context'] == 'Реабилитация'
mask_exoskelet = data['context'] == 'Exoskelet'
mask_Sofia = data['context'] == 'Sofia'
mask_apway = data['context'] == 'apway' 
mask_others = ~(mask_covid | mask_autism | mask_rehab | mask_exoskelet | mask_Sofia)
particular_column = 'old';

masks = (mask_covid, mask_autism, mask_rehab, mask_exoskelet, mask_Sofia, mask_apway, mask_others) 



C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2844240806.py:28: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show() # чтобы при каждой итерации окна не накапливались. сам график при этом не исчезает
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2844240806.py:28: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show() # чтобы при каждой итерации окна не накапливались. сам график при этом не исчезает
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2844240806.py:28: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show() # чтобы при каждой итерации окна не накапливались. сам график при этом не исчезает
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2844240806.py:28: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show() # чтобы при каж

C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2844240806.py:28: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show() # чтобы при каждой итерации окна не накапливались. сам график при этом не исчезает
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2844240806.py:28: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show() # чтобы при каждой итерации окна не накапливались. сам график при этом не исчезает
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2844240806.py:28: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show() # чтобы при каждой итерации окна не накапливались. сам график при этом не исчезает
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2844240806.py:28: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show() # чтобы при каж

# Построение графиков общего поведения точек

1) Через скользящее среднее (окно идёт по фиксированному количеству точек)

Находится средний x в окне. Находится средний y в окне. Находится стандартное отклонение y

In [39]:
def average_calculation(x, y, k):
    y_smoothed = []
    y_std = []
    x_smoothed = []
    for i in range (len(x) - k + 1):
        window_x = x[i : i + k]
        window_y = y[i : i + k]
        y_smoothed.append(np.mean(window_y)) #вычисляет среднее арифмитическое np.array и список можно передать
        y_std.append(np.std(window_y)) #вычисляет стандартное отклонение в массиве
        x_smoothed.append(np.mean(window_x))
    return np.array(x_smoothed), np.array(y_smoothed), np.array(y_std)

def fixed_window_plot(mask, color): #построить график через скользящее среднее
    # выделяем значения, .values преобразует данные в массив numpy
    x = data['old'][mask].values
    y = data[data.columns[i]][mask].values

    # Сортировка по x для корректного расчета скользящего окна
    sorted_indices = np.argsort(x)  #вернет индексы элеентов массива x, если значения отсорт по возрастанию
    x_sorted = x[sorted_indices]  #отсорт по возрастанию
    y_sorted = y[sorted_indices]  #сортируется так что каждые y соответствует отсортированному x

    x_sm, y_sm, std = average_calculation(x_sorted, y_sorted, number_of_points)
    #строим линию по получившимся точкам
    plt.plot(x_sm, y_sm, color=color, linewidth=2)
    
    plt.fill_between(
            x_sm,
            y_sm - std,
            y_sm + std,
            color=color,
            alpha=0.2,# чем меньше значение тем больше просвечивает т.е больше прозрачность
        )


In [40]:
number_of_points = 20 #колво точек для вычисления среднего значения (окно)

for i in range(9, len(data.columns)-1):
    if (data.columns[i] != particular_column and data.columns[i] != 'person' and data.columns[i] != 'Q_common'): 
        
        #plt.figure(figsize=(16,9), dpi= 100)
        plt.figure(figsize=(fig_width, fig_height), dpi=300)
        
        sns.scatterplot(
        data=data[mask_Sofia], 
        x=data['old'][mask_Sofia] + np.random.uniform(-0.5, 0.5, size=len(data[mask_Sofia])),# передается как данные
        y=data.columns[i],#передаётся как название солбца
        label='София',
        color=color_dict[0]
    )
        fixed_window_plot(mask_Sofia, color_dict[0])
         
        sns.scatterplot(
        data=data[mask_covid], 
        x=data['old'][mask_covid] + np.random.uniform(-0.5, 0.5, size=len(data[mask_covid])), 
        y=data.columns[i],
        label='Ковид',
        color=color_dict[2]
    )
        fixed_window_plot(mask_covid, color_dict[2])
        
        sns.scatterplot(
        data=data[mask_autism], 
        x=data['old'][mask_autism] + np.random.uniform(-0.5, 0.5, size=len(data[mask_autism])), 
        y=data.columns[i],
        label='Аутизм',
        color='black'
    )
        fixed_window_plot(mask_autism, 'black')

        sns.scatterplot(
        data=data[mask_rehab], 
        x=data['old'][mask_rehab] + np.random.uniform(-0.5, 0.5, size=len(data[mask_rehab])), 
        y=data.columns[i], 
        label='Рехаб',
        color='orange'
    )
        fixed_window_plot(mask_rehab, 'orange')
        
        sns.scatterplot(
        data=data[mask_exoskelet], 
        x=data['old'][mask_exoskelet] + np.random.uniform(-0.5, 0.5, size=len(data[mask_exoskelet])), 
        y=data.columns[i],
        label='Экзоскелет',
        color=color_dict[5]
    )
        fixed_window_plot(mask_exoskelet, color_dict[5])
        
        mask_others = ~(mask_covid | mask_autism | mask_rehab | mask_exoskelet | mask_Sofia)
        sns.scatterplot(
        data=data[mask_others],
        x=data['old'][mask_others] + 
        np.random.uniform(-0.5, 0.5, size=len(data[mask_others])),
        y=data.columns[i],
        label='Другие',
        color=color_dict[6]
    )
        fixed_window_plot(mask_others, color_dict[6])
        
        if i == 9:
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))# сдвиг легенды за график
            plt.xlabel('Возраст, 8 групп по 10 лет (года)', fontsize=14, fontweight='light')
            plt.ylabel('Продолжительность теста', fontsize=14, fontweight='light')
            plt.title('а', loc='left')
        else:
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))# сдвиг легенды за график
            plt.xlabel('Age, 8 groups for each 10 years (years)', fontsize=14, fontweight='light')
            plt.ylabel(data.columns[i], fontsize=14, fontweight='light')
        '''
        if (i in particular_graphs):
            column_number = str(i).zfill(2)
            plt.savefig(f'D:/AI/HIstogram/smoothed_plots/fixed_window/particular/plot_{column_number}_{data.columns[i]}_vs_{particular_column}.png', format='png', dpi=600)
        else:
            column_number = str(i).zfill(2)
            plt.savefig(f'D:/AI/HIstogram/smoothed_plots/fixed_window/plot_{column_number}_{data.columns[i]}_vs_{particular_column}.png', format='png', dpi=600)
        '''
        
        plt.show()
        plt.close()
        

C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2516228065.py:83: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2516228065.py:83: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2516228065.py:83: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2516228065.py:83: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2516228065.py:83: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2516228065.py:83: UserWarning:

2)Скользящее окно(окно идёт по значению возраста)

Находится средний x в окне. Находится средний y в окне. Находится стандартное отклонение y

In [41]:
def old_window_plot(data, mask, color, y):
    # выделяем значения, .values преобразует данные в массив numpy
    x = data['old'][mask].values
    y = data[y][mask].values

    # Сортировка по x для корректного расчета скользящего окна
    sorted_indices = np.argsort(x)  #вернет индексы элеентов массива x, если значения отсорт по возрастанию
    x_sorted = x[sorted_indices]  #отсорт по возрастанию
    y_sorted = y[sorted_indices]  #сортируется так что каждые y соответствует отсортированному x
    
    smoothed_mean = []
    smoothed_std = []
    x_smoothed = []
    half = window_size / 2

    for i in range (len(x_sorted)):
        central_point = x_sorted[i]
        left_board = central_point - half
        right_board = central_point + half
        
        mask = (x_sorted < right_board) & (x_sorted > left_board)
        window_x = x_sorted[mask]
        window_y = y_sorted[mask]
        
        smoothed_mean.append(np.mean(window_y))
        smoothed_std.append(np.std(window_y))
        x_smoothed.append(np.mean(window_x))  # Среднее значение x в окне
   
    
    plt.plot(x_smoothed, smoothed_mean, color=color, linewidth=2)
    plt.fill_between(
            x_smoothed,
            np.array(smoothed_mean) - np.array(smoothed_std),
            np.array(smoothed_mean) + np.array(smoothed_std),
            color=color,
            alpha=0.2,# чем меньше значение тем больше просвечивает т.е больше прозрачность
        )


In [42]:
window_size = 10 # (окно)

for i in range(9, len(data.columns)-1):
    if (data.columns[i] != particular_column and data.columns[i] != 'person' and data.columns[i] != 'Q_common'): 
        
        #plt.figure(figsize=(16,9), dpi= 100)
        plt.figure(figsize=(fig_width, fig_height), dpi=300)
        y = data.columns[i]
        
        sns.scatterplot(
        data=data[mask_Sofia], 
        x=data['old'][mask_Sofia] + np.random.uniform(-0.5, 0.5, size=len(data[mask_Sofia])),# передается как данные
        y=data.columns[i],#передаётся как название солбца
        label='София',
        color=color_dict[0]
    )
        old_window_plot(data, mask_Sofia, color_dict[0], y)
         
        sns.scatterplot(
        data=data[mask_covid], 
        x=data['old'][mask_covid] + np.random.uniform(-0.5, 0.5, size=len(data[mask_covid])), 
        y=data.columns[i],
        label='Ковид',
        color=color_dict[2]
    )
        old_window_plot(data, mask_covid, color_dict[2], y)
        
        sns.scatterplot(
        data=data[mask_autism], 
        x=data['old'][mask_autism] + np.random.uniform(-0.5, 0.5, size=len(data[mask_autism])), 
        y=data.columns[i],
        label='Аутизм',
        color='black'
    )
        old_window_plot(data, mask_autism, 'black', y)

        sns.scatterplot(
        data=data[mask_rehab], 
        x=data['old'][mask_rehab] + np.random.uniform(-0.5, 0.5, size=len(data[mask_rehab])), 
        y=data.columns[i], 
        label='Рехаб',
        color='orange'
    )
        old_window_plot(data, mask_rehab, 'orange', y)
        
        sns.scatterplot(
        data=data[mask_exoskelet], 
        x=data['old'][mask_exoskelet] + np.random.uniform(-0.5, 0.5, size=len(data[mask_exoskelet])), 
        y=data.columns[i],
        label='Экзоскелет',
        color=color_dict[5]
    )
        old_window_plot(data, mask_exoskelet, color_dict[5], y)
        
        mask_others = ~(mask_covid | mask_autism | mask_rehab | mask_exoskelet | mask_Sofia)
        sns.scatterplot(
        data=data[mask_others],
        x=data['old'][mask_others] + 
        np.random.uniform(-0.5, 0.5, size=len(data[mask_others])),
        y=data.columns[i],
        label='Другие',
        color=color_dict[6]
    )
        old_window_plot(data, mask_others, color_dict[6], y)
        
        
        if i == 9:
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))# сдвиг легенды за график
            plt.xlabel('Возраст, 8 групп по 10 лет (года)', fontsize=14, fontweight='light')
            plt.ylabel('Продолжительность теста', fontsize=14, fontweight='light')
            plt.title('в', loc='left')
        else:
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))# сдвиг легенды за график
            plt.xlabel('Age, 8 groups for each 10 years (years)', fontsize=14, fontweight='light')
            plt.ylabel(data.columns[i], fontsize=14, fontweight='light')
        
        '''
        if (i in particular_graphs):
            column_number = str(i).zfill(2)
            plt.savefig(f'D:/AI/HIstogram/smoothed_plots/old_window/particular/plot_{column_number}_{data.columns[i]}_vs_{particular_column}.png', format='png', dpi=600)
        else:
            column_number = str(i).zfill(2)
            plt.savefig(f'D:/AI/HIstogram/smoothed_plots/old_window/plot_{column_number}_{data.columns[i]}_vs_{particular_column}.png', format='png', dpi=600)
        '''
        plt.show()
        plt.close()
        

C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\3740381450.py:85: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\3740381450.py:85: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\3740381450.py:85: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\3740381450.py:85: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\3740381450.py:85: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\3740381450.py:85: UserWarning:

3) Через гауссово взвешанное

Для каждого a = x находятся веса остальных x. С помощью этих весов находится y и стандартное отклонение

In [43]:
def gaussian_smoothed_plot(data, mask, color, y_name, x_name, sigma):
    # выделяем значения, .values преобразует данные в массив numpy
    x = data[x_name][mask].values
    y = data[y_name][mask].values

    # Сортировка по x для корректного расчета скользящего окна
    sorted_indices = np.argsort(x)  #вернет индексы элементов массива x, если значения отсорт по возрастанию
    x_sorted = x[sorted_indices]  #отсорт по возрастанию
    y_sorted = y[sorted_indices]  #сортируется так что каждые y соответствует отсортированному x
    
    smoothed_mean = []
    smoothed_std = []
    x_smoothed = []
    
    for a in x_sorted:
        weights = np.exp(-((x_sorted - a) ** 2) / (2 * sigma ** 2)) # массив с весами для каждого x
        weighted_mean = np.sum(weights * y_sorted) / np.sum(weights)  #взвешанное среднее для y который соответствует a
        weighted_std = np.sqrt(np.sum(weights * (y_sorted - weighted_mean) ** 2) / np.sum(weights))
        
        # Сохраняем результаты
        x_smoothed.append(a)
        smoothed_mean.append(weighted_mean)
        smoothed_std.append(weighted_std)
        
    plt.plot(x_smoothed, smoothed_mean, color=color, linewidth=2)
    plt.fill_between(
            x_smoothed,
            np.array(smoothed_mean) - np.array(smoothed_std),
            np.array(smoothed_mean) + np.array(smoothed_std),
            color=color,
            alpha=0.2,# чем меньше значение тем больше просвечивает т.е больше прозрачность
        )

In [44]:
sigma = 1

for i in range(9, len(data.columns)-1):
    if (data.columns[i] != particular_column and data.columns[i] != 'person' and data.columns[i] != 'Q_common'): 
        
        #plt.figure(figsize=(16,9), dpi= 100)
        plt.figure(figsize=(fig_width, fig_height), dpi=300)
        
        y = data.columns[i]
        
        sns.scatterplot(
        data=data[mask_Sofia], 
        x=data['old'][mask_Sofia] + np.random.uniform(-0.5, 0.5, size=len(data[mask_Sofia])),# передается как данные
        y=data.columns[i],#передаётся как название солбца
        label='София',
        color=color_dict[0]
    )
        gaussian_smoothed_plot(data, mask_Sofia, color_dict[0], y, 'old', sigma)
         
        sns.scatterplot(
        data=data[mask_covid], 
        x=data['old'][mask_covid] + np.random.uniform(-0.5, 0.5, size=len(data[mask_covid])), 
        y=data.columns[i],
        label='Ковид',
        color=color_dict[2]
    )
        gaussian_smoothed_plot(data, mask_covid, color_dict[2], y, 'old', sigma)
        
        sns.scatterplot(
        data=data[mask_autism], 
        x=data['old'][mask_autism] + np.random.uniform(-0.5, 0.5, size=len(data[mask_autism])), 
        y=data.columns[i],
        label='Аутизм',
        color='black'
    )
        gaussian_smoothed_plot(data, mask_autism, 'black', y, 'old', sigma)

        sns.scatterplot(
        data=data[mask_rehab], 
        x=data['old'][mask_rehab] + np.random.uniform(-0.5, 0.5, size=len(data[mask_rehab])), 
        y=data.columns[i], 
        label='Рехаб',
        color='orange'
    )
        gaussian_smoothed_plot(data, mask_rehab, 'orange', y, 'old', sigma)
        
        sns.scatterplot(
        data=data[mask_exoskelet], 
        x=data['old'][mask_exoskelet] + np.random.uniform(-0.5, 0.5, size=len(data[mask_exoskelet])), 
        y=data.columns[i],
        label='Экзоскелет',
        color=color_dict[5]
    )
        gaussian_smoothed_plot(data, mask_exoskelet, color_dict[5], y, 'old', sigma)
        
        mask_others = ~(mask_covid | mask_autism | mask_rehab | mask_exoskelet | mask_Sofia)
        sns.scatterplot(
        data=data[mask_others],
        x=data['old'][mask_others] + 
        np.random.uniform(-0.5, 0.5, size=len(data[mask_others])),
        y=data.columns[i],
        label='Другие',
        color=color_dict[6]
    )
        gaussian_smoothed_plot(data, mask_others, color_dict[6], y, 'old', sigma)
        
        if i == 9:
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))# сдвиг легенды за график
            plt.xlabel('Возраст, 8 групп по 10 лет (года)', fontsize=14, fontweight='light')
            plt.ylabel('Продолжительность теста', fontsize=14, fontweight='light')
            plt.title('б', loc='left')
        else:
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))# сдвиг легенды за график
            plt.xlabel('Age, 8 groups for each 10 years (years)', fontsize=14, fontweight='light')
            plt.ylabel(data.columns[i], fontsize=14, fontweight='light')
        
        
        '''
        if (i in particular_graphs):
            column_number = str(i).zfill(2)
            plt.savefig(f'D:/AI/HIstogram/smoothed_plots/gaussian_smoothed/particular/plot_{column_number}_{data.columns[i]}_vs_{particular_column}.png', format='png', dpi=600)
        else:
            column_number = str(i).zfill(2)
            plt.savefig(f'D:/AI/HIstogram/smoothed_plots/gaussian_smoothed/plot_{column_number}_{data.columns[i]}_vs_{particular_column}.png', format='png', dpi=600)
        '''
        plt.show()
        plt.close()
        

C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2459904345.py:86: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2459904345.py:86: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2459904345.py:86: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2459904345.py:86: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2459904345.py:86: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\2459904345.py:86: UserWarning:

# Построение графиков KDE для разных параметров

In [45]:
for i in range(9,len(data.columns)-1):
    if (data.columns[i] != particular_column and data.columns[i] != 'person') :
        plt.figure(figsize=(16,9), dpi= 100)
        sns.kdeplot(data=data[mask_Sofia][data.columns[i]],color='pink',label='Sofia')
        sns.kdeplot(data=data[mask_apway][data.columns[i]],color='black',label='apway')
        sns.kdeplot(data=data[mask_covid][data.columns[i]],color='red',label='covid')
        sns.kdeplot(data=data[mask_autism][data.columns[i]],color='green',label='autism')
        sns.kdeplot(data=data[mask_rehab][data.columns[i]],color='brown',label='rehab')
        sns.kdeplot(data=data[mask_exoskelet][data.columns[i]],color="orange",label='exoskelet')
        sns.kdeplot(data=data[~(mask_covid | mask_autism | mask_rehab | mask_exoskelet | mask_Sofia | mask_apway)][data.columns[i]], color='blue', label='Others')
        
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))# сдвиг легенды за график
        plt.ylabel(data.columns[i], fontsize=14, fontweight='light')
       
        
        plt.show()
        plt.close()
        


C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\677635422.py:16: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\677635422.py:16: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\677635422.py:16: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\677635422.py:8: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(data=data[mask_rehab][data.columns[i]],color='brown',label='rehab')
C:\Users\kirya\AppData\Local\Temp\ipykernel_24580\677635422.py:16: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\U

TypeError: The x variable is categorical, but one of ['numeric', 'datetime'] is required